# Learning & Validation Curves

The [metrics deep-dive](metrics-deep-dive.ipynb) told you *how good* a model is.
This chapter diagnoses *why* — is it **underfitting** (too simple, high bias) or
**overfitting** (too complex, high variance)? — and whether **more data** would
help. Two diagnostic plots answer these:

- a **learning curve** varies the *training-set size*;
- a **validation curve** varies a single *hyperparameter*.

They look similar but ask different questions — keeping them distinct is the
point of this chapter. We reuse `smartcore`'s breast-cancer data throughout.

In [ ]:
:dep smartcore = { version = "0.3", features = ["datasets"] }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::dataset::breast_cancer;
use smartcore::linear::logistic_regression::LogisticRegression;
use smartcore::metrics::accuracy;
use plotters::prelude::*;

// Load raw, then shuffle+split INDICES by hand (a fixed-seed Fisher-Yates) into a
// training pool and a held-out validation set. Working with indices lets us build
// arbitrary training subsets for the learning curve. Everything persisted here is
// a plain Vec (a type evcxr can keep across cells).
let (data, target, nf, train_idx, val_idx): (Vec<f32>, Vec<i32>, usize, Vec<usize>, Vec<usize>) = {
    let ds = breast_cancer::load_dataset();
    let (ns, nf) = (ds.num_samples, ds.num_features);
    let data = ds.data.clone();
    let target: Vec<i32> = ds.target.iter().map(|&v| v as i32).collect();
    let mut idx: Vec<usize> = (0..ns).collect();
    let mut seed = 42u64;
    for i in (1..ns).rev() {
        seed = seed.wrapping_mul(6364136223846793005).wrapping_add(1);
        let j = ((seed >> 33) as usize) % (i + 1);
        idx.swap(i, j);
    }
    let n_val = ns / 3;
    (data, target, nf, idx[n_val..].to_vec(), idx[..n_val].to_vec())
};
println!("training pool = {}, validation = {}", train_idx.len(), val_idx.len());

In [ ]:
// Helpers: build a DenseMatrix / label vector from a set of row indices.
fn submatrix(data: &[f32], nf: usize, rows: &[usize]) -> DenseMatrix<f32> {
    let mut buf = Vec::with_capacity(rows.len() * nf);
    for &r in rows { for j in 0..nf { buf.push(data[r * nf + j]); } }
    DenseMatrix::new(rows.len(), nf, buf, false)
}
fn sublabels(target: &[i32], rows: &[usize]) -> Vec<i32> {
    rows.iter().map(|&r| target[r]).collect()
}
println!("helpers ready");

## Learning curve — does more data help?

Train the model on growing slices of the training pool; at each size, score it on
that training slice **and** on the fixed validation set. Reading the two curves:

- a large **gap** (train ≫ validation) → **high variance / overfitting** — more
  data or regularization should help;
- both curves **plateau low and together** → **high bias / underfitting** — more
  data won't help; you need a more flexible model or better features.

In [ ]:
let xval = submatrix(&data, nf, &val_idx);
let yval = sublabels(&target, &val_idx);

let sizes: Vec<usize> = {
    let maxn = train_idx.len();
    let step = (maxn / 8).max(1);
    let mut v = vec![];
    let mut s = 20usize;
    while s < maxn { v.push(s); s += step; }
    v.push(maxn);
    v
};

let mut tr_curve: Vec<(f64, f64)> = vec![];
let mut va_curve: Vec<(f64, f64)> = vec![];
for &k in &sizes {
    let rows = &train_idx[..k];
    let xk = submatrix(&data, nf, rows);
    let yk = sublabels(&target, rows);
    let model = LogisticRegression::fit(&xk, &yk, Default::default()).unwrap();
    tr_curve.push((k as f64, accuracy(&yk, &model.predict(&xk).unwrap())));
    va_curve.push((k as f64, accuracy(&yval, &model.predict(&xval).unwrap())));
}

evcxr_figure((580, 420), |root| {
    root.fill(&WHITE)?;
    let x0 = sizes[0] as f64;
    let x1 = *sizes.last().unwrap() as f64;
    let mut chart = ChartBuilder::on(&root)
        .caption("learning curve (logistic regression)", ("sans-serif", 16))
        .margin(10).x_label_area_size(36).y_label_area_size(44)
        .build_cartesian_2d(x0..x1, 0.80f64..1.01f64)?;
    chart.configure_mesh().x_desc("training samples").y_desc("accuracy").draw()?;
    chart.draw_series(LineSeries::new(tr_curve.clone(), BLUE.stroke_width(2)))?
        .label("train").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], BLUE));
    chart.draw_series(tr_curve.iter().map(|p| Circle::new(*p, 3, BLUE.filled())))?;
    chart.draw_series(LineSeries::new(va_curve.clone(), RED.stroke_width(2)))?
        .label("validation").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], RED));
    chart.draw_series(va_curve.iter().map(|p| Circle::new(*p, 3, RED.filled())))?;
    chart.configure_series_labels().position(SeriesLabelPosition::LowerRight)
        .background_style(WHITE.mix(0.85)).border_style(BLACK).draw()?;
    Ok(())
})

The train curve starts near-perfect (easy to fit a handful of points) and
settles as data grows; the validation curve climbs toward it. A persistent gap
would point to variance — the concrete fix is the **regularization** from the
Regression chapters (Ridge/Lasso), which trades a little training fit for better
generalization.

## Validation curve — how complex should the model be?

Now hold the data fixed and vary a **hyperparameter** instead. A decision tree's
`max_depth` is the classic knob: too shallow underfits, too deep overfits by
memorising the training set. The train/validation gap widening as depth grows is
overfitting made visible.

In [ ]:
use smartcore::tree::decision_tree_classifier::{DecisionTreeClassifier, DecisionTreeClassifierParameters};

let xtr = submatrix(&data, nf, &train_idx);
let ytr = sublabels(&target, &train_idx);
let xval = submatrix(&data, nf, &val_idx);
let yval = sublabels(&target, &val_idx);

let depths: Vec<u16> = (1..=12).collect();
let mut tr_curve: Vec<(f64, f64)> = vec![];
let mut va_curve: Vec<(f64, f64)> = vec![];
for &d in &depths {
    let params = DecisionTreeClassifierParameters::default().with_max_depth(d);
    let model = DecisionTreeClassifier::fit(&xtr, &ytr, params).unwrap();
    tr_curve.push((d as f64, accuracy(&ytr, &model.predict(&xtr).unwrap())));
    va_curve.push((d as f64, accuracy(&yval, &model.predict(&xval).unwrap())));
}

evcxr_figure((580, 420), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("validation curve (tree max_depth)", ("sans-serif", 16))
        .margin(10).x_label_area_size(36).y_label_area_size(44)
        .build_cartesian_2d(1f64..12f64, 0.80f64..1.01f64)?;
    chart.configure_mesh().x_desc("max_depth").y_desc("accuracy").draw()?;
    chart.draw_series(LineSeries::new(tr_curve.clone(), BLUE.stroke_width(2)))?
        .label("train").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], BLUE));
    chart.draw_series(tr_curve.iter().map(|p| Circle::new(*p, 3, BLUE.filled())))?;
    chart.draw_series(LineSeries::new(va_curve.clone(), RED.stroke_width(2)))?
        .label("validation").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], RED));
    chart.draw_series(va_curve.iter().map(|p| Circle::new(*p, 3, RED.filled())))?;
    chart.configure_series_labels().position(SeriesLabelPosition::LowerRight)
        .background_style(WHITE.mix(0.85)).border_style(BLACK).draw()?;
    Ok(())
})

## Learning vs validation curves — don't conflate them

| Curve | X-axis varies | Answers |
| --- | --- | --- |
| **Learning curve** | training-set **size** | Would more data help? Bias vs. variance at fixed complexity. |
| **Validation curve** | one **hyperparameter** | What complexity generalizes best? Where does overfitting start? |

Both use the train-vs-validation gap as the diagnostic — but tuning the *knob*
(validation curve) is the job of the [Optimization](../05b-optimization/hyperparameter-search.ipynb)
chapter's hyperparameter search, done systematically instead of by eye.

Next: back to modelling, now equipped to evaluate and diagnose every model you
build.